# Drift plus noise

Random walk in log price with constant drift: `dlog p = mu dt + sigma dW`. The simplest trending case.

Fantasma simulates **log returns** (log price differences): one row per step, labelled at the end of the step. `fm.to_prices`, `fm.to_log_prices` and `fm.to_returns` turn them into other quantities.

Edit the parameters cell and re-run all: the same parameters and seed always give the same paths.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import fantasma as fm

## Parameters

Model parameters are annualised; `dt` is the step length over a 365.25-day year.

In [ ]:
SEED = 42
N_PATHS = 5
START = "2024-01-01"
FREQ = "1d"  # Almanac frequency ("1m", "10m", "1h", "1d") or any pandas offset
N_STEPS = 365 * 5

MU = 0.10
SIGMA = 0.2
INITIAL_PRICE = 100.0
ASSET = "DN"

## Simulate

In [ ]:
model = fm.DriftPlusNoise(mu=MU, sigma=SIGMA, asset=ASSET, initial_price=INITIAL_PRICE)
model

In [ ]:
paths = fm.simulate(
    model,
    start=START,
    freq=FREQ,
    n_steps=N_STEPS,
    seed=SEED,
    n_paths=N_PATHS,
    return_hidden=True,
)
log_returns, hidden = paths[0]  # log returns; hidden state on the full grid
log_returns.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for i, (path_log_returns, _) in enumerate(paths):
    path_prices = fm.to_prices(path_log_returns)  # start included, from the recorded start price
    for asset in path_prices.columns:
        ax.plot(path_prices.index, path_prices[asset], lw=0.8, label=f"path {i} {asset}")
ax.set_title("Drift plus noise")
ax.set_ylabel("price")
if len(paths) * paths[0][0].shape[1] <= 10:
    ax.legend(fontsize=8)
plt.show()

## Check: drift and volatility

The drift estimate's standard error is `SIGMA / sqrt(T)` with `T` in years, so it is noisy unless the path is long.

In [ ]:
dt = fm.RegularGrid(START, FREQ, n_steps=1).dt
r = log_returns[ASSET].to_numpy()
mu_hat, sigma_hat = r.mean() / dt, r.std(ddof=1) / np.sqrt(dt)
years = len(r) * dt
print(f"mu    = {MU:.4f}, estimate = {mu_hat:.4f}, s.e. = {SIGMA / np.sqrt(years):.4f}")
print(f"sigma = {SIGMA:.4f}, estimate = {sigma_hat:.4f}, s.e. = {SIGMA / np.sqrt(2 * len(r)):.4f}")